In [0]:
# Schema:
# order_id         STRING
# item_id          STRING
# restaurant_id    STRING
# order_timestamp  TIMESTAMP
# order_date       DATE
# item_name        STRING
# category         STRING
# quantity         INT
# unit_price       DECIMAL(10,2)
# subtotal         DECIMAL(10,2)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:

_schema = ArrayType(
    StructType([
    StructField("item_id", StringType()),
    StructField("name", StringType()),
    StructField("category", StringType()),
    StructField("quantity", IntegerType()),
    StructField("unit_price", DecimalType(10,2)),
    StructField("subtotal", DecimalType(10,2))
])
) 

df_fact_order_items = (
    spark.table("01_bronze.orders")
    .withColumn("order_timestamp", to_timestamp(col("order_timestamp")))
    .withColumn("order_date", to_date(col('order_timestamp')))
    .withColumn("item_parsed", from_json(col("items"), _schema))
    .withColumn("items", explode(col("item_parsed")))
    .select(
        "order_id",
        col("items.item_id").alias("item_id"),
        "restaurant_id",
        "order_timestamp",
        "order_date",
        col("items.name").alias("item_name"),
        col("items.quantity").alias("quantity"),
        col("items.unit_price").cast("decimal(10,2)").alias("unit_price"),
        col("items.subtotal").cast("decimal(10,2)").alias("sub_total")
    )
)


display(df_fact_order_items)
